In [1]:
import pandas as pd
import glob 
import subprocess
import json
import os

In [2]:
data_df = pd.read_csv("../data/merged_data.csv")[["game", "position", "recordingIds", "recordingsFolder", "sampleId", "muteEvents", "discussion_happened"]]

collector = []
for index, row in data_df.iterrows():
    game = row["game"]
    position = row["position"]
    recordingIdsStr = row['recordingIds']
    if recordingIdsStr == "missing":
        continue
    recordingIds = json.loads(recordingIdsStr.strip("\"").replace("\'", "\""))
    # print(game, position, recordingIds)
    
    for recordingId in recordingIds:
        collector.append({
            "game": game,
            "position": position,
            "recordingId": recordingId,
            "recordingsFolder": row["recordingsFolder"],
            "sampleId": row["sampleId"],
            "muteEvents": row["muteEvents"],
            "discussion_happened": row["discussion_happened"]
        })
recording_ids_df = pd.DataFrame(collector).set_index("recordingId")
recording_ids_df
        

/var/folders/2l/3kmvzs0s1sggnw_9sl0ls8080000gn/T/ipykernel_32051/2103093167.py:1: DtypeWarning: Columns (28,53,55,576) have mixed types. Specify dtype option on import or set low_memory=False.
  data_df = pd.read_csv("../data/merged_data.csv")[["game", "position", "recordingIds", "recordingsFolder", "sampleId", "muteEvents", "discussion_happened"]]


,game,position,recordingsFolder,sampleId,muteEvents,discussion_happened
recordingId,,,,,,
252fcba0-f6df-41af-ae27-3516965e50b4,0,0,20241001_1844_ctt_w1V35R4E,667c2bfe-8401-4272-9ea6-44f0aef7c38c,"[[""muted-video"", 4.486], [""muted-audio"", 5.469...",True
c3eac583-9b28-4756-8cc6-947d2fc09e31,0,1,20241001_1844_ctt_w1V35R4E,0be567b7-cda5-42eb-9098-c0187b0225c3,"[[""muted-video"", 4.881], [""muted-audio"", 4.881...",True
1cfdf05f-0eb7-4311-bce8-fbaca8f229a5,1,0,20241004_1934_ctt_w1M7AWTB,3fe79b64-0703-4a0e-8389-878ba6f5a386,"[[""muted-video"", 5.506], [""muted-audio"", 5.506...",True
eb2785d2-ee20-488a-9bd8-d8da06724910,1,1,20241004_1934_ctt_w1M7AWTB,b9c46cfa-6b1d-4f62-8efa-2362ac93dfe1,"[[""muted-video"", 5.966], [""muted-audio"", 5.966...",True
299867da-d96e-4ad4-9f41-e3033ac95261,2,0,20240923_1531_CTT_W1GY0Y4B,2bd517af-2e4c-41f8-9dea-ee2eff4b2ca4,"[[""muted-audio"", 9.574], [""muted-video"", 11.44...",True
...,...,...,...,...,...,...
c243ef8e-bda5-4d4d-bc6d-29a306547dd8,513,1,20241217_1837_w2s19_BS9KEY,2cf11ea9-9d50-4508-9ad0-6d08eb9acefe,"[[""muted-video"", 4.197], [""muted-audio"", 4.197...",True
5ee1ac81-ad68-4d68-9bff-fe5e72f8fc1d,514,0,20241219_1734_w2s26_19WP2C,428b2f57-3e3b-4178-b884-5e305f24cce1,"[[""muted-video"", 5.992], [""muted-audio"", 5.992...",True
96ebfe1f-5512-4b85-8f0a-6213c3dee2db,514,1,20241219_1734_w2s26_19WP2C,b9be4fe4-079a-470e-ad45-b609bec439cd,"[[""muted-audio"", 5.511], [""muted-video"", 5.511...",True


In [3]:
files = glob.glob("recordings/**/*.webm")

collector = []
for file in files:
    if "normalized" in file: continue
    try:
        _, folder, filename = file.split("/")
        date, time, batch, group = folder.split("_")
        mtgStartTime, *sessionId, _, filetype, trackStartTime = filename.split("-")
        sessionId = "_".join(sessionId)
        game, position, sampleId = -1,-1,None
        for recordingId in recording_ids_df.index:
            if recordingId in file:
                game, position, sampleId, muteEvents, discussion_happened = recording_ids_df.loc[recordingId][["game", "position", "sampleId", "muteEvents", "discussion_happened"]].values
                break
        
        # game, position = recording_ids_df.loc[sessionId][["game", "position"]].values
        collector.append({
            "path": file,
            "folder": folder,
            "filename": filename,
            "date": date,
            "time": time,
            "mtgStartTime": mtgStartTime,
            "sessionId": sessionId,
            "batch": batch,
            "group": group,
            "filetype": filetype,
            "game": int(game),
            "position": int(position),
            "sampleId": sampleId,
            "muteEvents": muteEvents,
            "discussion_happened": discussion_happened
        })
        
    except Exception as e:
        print(f"Error processing file {file}: {e}")
        continue


recordings_df = pd.DataFrame(collector).sort_values("path").dropna(subset="sampleId").reset_index(drop=True)

pd.options.display.max_colwidth = 150
recordings_df

Error processing file recordings/20240815_2018_miniPiZ4VQ1R/1723755676733-9fd4b432-e039-453d-8380-5d6b369ed807-cam-audio-1723755677707.webm: not enough values to unpack (expected 4, got 3)
Error processing file recordings/20240815_2018_miniPiZ4VQ1R/1723755676733-9dfd15cc-20be-4158-a847-290b4518db5a-cam-audio-1723755677706.webm: not enough values to unpack (expected 4, got 3)
Error processing file recordings/20240815_2018_miniPiZ4VQ1R/1723755676733-9dfd15cc-20be-4158-a847-290b4518db5a-cam-video-1723755677708.webm: not enough values to unpack (expected 4, got 3)
Error processing file recordings/20240815_2018_miniPiZ4VQ1R/1723755676733-9fd4b432-e039-453d-8380-5d6b369ed807-cam-video-1723755677704.webm: not enough values to unpack (expected 4, got 3)
Error processing file recordings/test/1741225144124-d9f8d196-a5b5-4bb7-87b3-3616e706bccb-cam-video-1741225240765.webm: not enough values to unpack (expected 4, got 1)
Error processing file recordings/test/1741225144124-fb9f93ff-5e48-40a2-bb09-b

,path,folder,filename,date,time,mtgStartTime,sessionId,batch,group,filetype,game,position,sampleId,muteEvents,discussion_happened
0,recordings/20240919_1914_CT_top0EEWMB/1726776113035-3d09dc36-e808-4419-af86-ff9ab6f8da35-cam-audio-1726776114008.webm,20240919_1914_CT_top0EEWMB,1726776113035-3d09dc36-e808-4419-af86-ff9ab6f8da35-cam-audio-1726776114008.webm,20240919,1914,1726776113035,3d09dc36_e808_4419_af86_ff9ab6f8da35,CT,top0EEWMB,audio,149,0,292f062a-4c79-4d4d-903a-686d21b602d3,"[[""muted-video"", 16.959], [""muted-audio"", 16.959], [""unmuted-audio"", 17.958]]",True
1,recordings/20240919_1914_CT_top0EEWMB/1726776113035-3d09dc36-e808-4419-af86-ff9ab6f8da35-cam-video-1726776114010.webm,20240919_1914_CT_top0EEWMB,1726776113035-3d09dc36-e808-4419-af86-ff9ab6f8da35-cam-video-1726776114010.webm,20240919,1914,1726776113035,3d09dc36_e808_4419_af86_ff9ab6f8da35,CT,top0EEWMB,video,149,0,292f062a-4c79-4d4d-903a-686d21b602d3,"[[""muted-video"", 16.959], [""muted-audio"", 16.959], [""unmuted-audio"", 17.958]]",True
2,recordings/20240919_1914_CT_top0EEWMB/1726776113035-9491dd61-941c-42ba-b3a0-92b9869649f1-cam-audio-1726776121625.webm,20240919_1914_CT_top0EEWMB,1726776113035-9491dd61-941c-42ba-b3a0-92b9869649f1-cam-audio-1726776121625.webm,20240919,1914,1726776113035,9491dd61_941c_42ba_b3a0_92b9869649f1,CT,top0EEWMB,audio,149,1,eb76771c-1722-448f-8539-decd0a97b038,"[[""muted-video"", 13.258], [""unmuted-video"", 16.258], [""muted-video"", 17.258], [""unmuted-video"", 20.259]]",True
3,recordings/20240919_1914_CT_topH7FS3D/1726776119989-2517b174-8a36-4710-8387-e22bcc52339e-cam-audio-1726776120977.webm,20240919_1914_CT_topH7FS3D,1726776119989-2517b174-8a36-4710-8387-e22bcc52339e-cam-audio-1726776120977.webm,20240919,1914,1726776119989,2517b174_8a36_4710_8387_e22bcc52339e,CT,topH7FS3D,audio,12,1,0f9d6583-d005-43b4-88c4-305cbebacfa5,"[[""muted-audio"", 4.291], [""muted-video"", 5.29], [""unmuted-audio"", 5.29], [""unmuted-video"", 5.29], [""muted-audio"", 10.29], [""muted-video"", 284.288]...",False
4,recordings/20240919_1914_CT_topH7FS3D/1726776119989-2517b174-8a36-4710-8387-e22bcc52339e-cam-video-1726776120979.webm,20240919_1914_CT_topH7FS3D,1726776119989-2517b174-8a36-4710-8387-e22bcc52339e-cam-video-1726776120979.webm,20240919,1914,1726776119989,2517b174_8a36_4710_8387_e22bcc52339e,CT,topH7FS3D,video,12,1,0f9d6583-d005-43b4-88c4-305cbebacfa5,"[[""muted-audio"", 4.291], [""muted-video"", 5.29], [""unmuted-audio"", 5.29], [""unmuted-video"", 5.29], [""muted-audio"", 10.29], [""muted-video"", 284.288]...",False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2294,recordings/20241219_1734_w2s26_WT9T50/1734631629271-fda80aa9-954a-4b2b-828d-ca31b7cc296a-cam-video-1734631780694.webm,20241219_1734_w2s26_WT9T50,1734631629271-fda80aa9-954a-4b2b-828d-ca31b7cc296a-cam-video-1734631780694.webm,20241219,1734,1734631629271,fda80aa9_954a_4b2b_828d_ca31b7cc296a,w2s26,WT9T50,video,409,1,32a46374-c7a2-4f9c-b6c8-e40954c4791f,"[[""muted-audio"", 14.721], [""muted-video"", 14.721], [""unmuted-audio"", 14.721], [""unmuted-video"", 15.721], [""muted-audio"", 83.311], [""muted-video"", ...",True
2295,recordings/20241219_1734_w2s26_ZK6YGD/1734631999615-1d3ce1f0-c7d4-4020-ba89-4dd962bf1982-cam-audio-1734632005066.webm,20241219_1734_w2s26_ZK6YGD,1734631999615-1d3ce1f0-c7d4-4020-ba89-4dd962bf1982-cam-audio-1734632005066.webm,20241219,1734,1734631999615,1d3ce1f0_c7d4_4020_ba89_4dd962bf1982,w2s26,ZK6YGD,audio,258,0,1cb175d2-7c44-4ca7-9bb0-7fd666d4462b,"[[""muted-audio"", 12.153], [""muted-video"", 12.153], [""unmuted-audio"", 12.153], [""unmuted-video"", 12.153]]",True
2296,recordings/20241219_1734_w2s26_ZK6YGD/1734631999615-1d3ce1f0-c7d4-4020-ba89-4dd962bf1982-cam-video-1734632005294.webm,20241219_1734_w2s26_ZK6YGD,1734631999615-1d3ce1f0-c7d4-4020-ba89-4dd962bf1982-cam-video-1734632005294.webm,20241219,1734,1734631999615,1d3ce1f0_c7d4_4020_ba89_4dd962bf1982,w2s26,ZK6YGD,video,258,0,1cb175d2-7c44-4ca7-9bb0-7fd666d4462b,"[[""muted-audio"", 12.153], [""mute

In [4]:
# create test recordings dataframe

test_recordings_df = pd.DataFrame(collector)
test_recordings_df = test_recordings_df[test_recordings_df["folder"] == "20250301_0011_demo_QPK9NM"].reset_index(drop=True)

test_positions = {
    "4395b77a_c5d1_482a_b413_ca212d06fc0c":1,
    "dc777556_1039_4be7_a46b_00778e3479cd":0,
    "353873af_6c54_4510_bbcd_bd275feb611c":1

}

test_recordings_df["position"] = test_recordings_df["sessionId"].map(test_positions)
test_recordings_df

,path,folder,filename,date,time,mtgStartTime,sessionId,batch,group,filetype,game,position,sampleId,muteEvents,discussion_happened
0,recordings/20250301_0011_demo_QPK9NM/1740787968485-dc777556-1039-4be7-a46b-00778e3479cd-cam-audio-1740787968762.webm,20250301_0011_demo_QPK9NM,1740787968485-dc777556-1039-4be7-a46b-00778e3479cd-cam-audio-1740787968762.webm,20250301,0011,1740787968485,dc777556_1039_4be7_a46b_00778e3479cd,demo,QPK9NM,audio,-1,0,None,"[[""muted-audio"", 5.738], [""muted-video"", 5.738], [""unmuted-audio"", 5.738], [""unmuted-video"", 5.738]]",True
1,recordings/20250301_0011_demo_QPK9NM/1740787968485-353873af-6c54-4510-bbcd-bd275feb611c-cam-video-1740788033829.webm,20250301_0011_demo_QPK9NM,1740787968485-353873af-6c54-4510-bbcd-bd275feb611c-cam-video-1740788033829.webm,20250301,0011,1740787968485,353873af_6c54_4510_bbcd_bd275feb611c,demo,QPK9NM,video,-1,1,None,"[[""muted-audio"", 5.738], [""muted-video"", 5.738], [""unmuted-audio"", 5.738], [""unmuted-video"", 5.738]]",True
2,recordings/20250301_0011_demo_QPK9NM/1740787968485-353873af-6c54-4510-bbcd-bd275feb611c-cam-audio-1740788033827.webm,20250301_0011_demo_QPK9NM,1740787968485-353873af-6c54-4510-bbcd-bd275feb611c-cam-audio-1740788033827.webm,20250301,0011,1740787968485,353873af_6c54_4510_bbcd_bd275feb611c,demo,QPK9NM,audio,-1,1,None,"[[""muted-audio"", 5.738], [""muted-video"", 5.738], [""unmuted-audio"", 5.738], [""unmuted-video"", 5.738]]",True
3,recordings/20250301_0011_demo_QPK9NM/1740787968485-4395b77a-c5d1-482a-b413-ca212d06fc0c-cam-audio-1740787968764.webm,20250301_0011_demo_QPK9NM,1740787968485-4395b77a-c5d1-482a-b413-ca212d06fc0c-cam-audio-1740787968764.webm,20250301,0011,1740787968485,4395b77a_c5d1_482a_b413_ca212d06fc0c,demo,QPK9NM,audio,-1,1,None,"[[""muted-audio"", 5.738], [""muted-video"", 5.738], [""unmuted-audio"", 5.738], [""unmuted-video"", 5.738]]",True
4,recordings/20250301_0011_demo_QPK9NM/1740787968485-dc777556-1039-4be7-a46b-00778e3479cd-cam-video-1740787968767.webm,20250301_0011_demo_QPK9NM,1740787968485-dc777556-1039-4be7-a46b-00778e3479cd-cam-video-1740787968767.webm,20250301,0011,1740787968485,dc777556_1039_4be7_a46b_00778e3479cd,demo,QPK9NM,video,-1,0,None,"[[""muted-audio"", 5.738], [""muted-video"", 5.738], [""unmuted-audio"", 5.738], [""unmuted-video"", 5.738]]",True
5,recordings/20250301_0011_demo_QPK9NM/1740787968485-4395b77a-c5d1-482a-b413-ca212d06fc0c-cam-video-1740787968759.webm,20250301_0011_demo_QPK9NM,1740787968485-4395b77a-c5d1-482a-b413-ca212d06fc0c-cam-video-1740787968759.webm,20250301,0011,1740787968485,4395b77a_c5d1_482a_b413_ca212d06fc0c,demo,QPK9NM,video,-1,1,None,"[[""muted-audio"", 5.738], [""muted-video"", 5.738], [""unmuted-audio"", 5.738], [""unmuted-video"", 5.738]]",True


In [18]:
"""
# Extracting Media Start Times

## Purpose:
This cell extracts the start time of each media file (video or audio) in the dataset. 
Since recordings may begin at different times due to participant delays or recording lags, 
we need to align them properly when merging. The start time will be used in later steps 
to correctly synchronize the media files.

## Process:
1. **`get_media_start_time(media_path, stream_type="v:0")`**:
   - Uses `ffprobe` to extract the start time of a given media file.
   - Supports both video (`v:0`) and audio (`a:0`) streams.
   - Returns the start time as a float (or `0` if unavailable).

2. **`apply_start_time(row)`**:
   - Determines whether the media file is audio or video.
   - Calls `get_media_start_time()` with the correct stream type.
   - Returns the extracted start time or `None` if an error occurs.

3. **Applying to the dataset**:
   - The `start_time` column is computed for each media file in `recordings_df`.
   - This ensures that each file's relative position in time is accounted for when 
     synchronizing and merging media files later in the pipeline.

## Why This Matters:
Without correcting for differing start times, the final merged video and audio 
could be misaligned, leading to desynchronized playback.
"""


def get_media_start_time(media_path, stream_type="v:0"):  
    """Extracts start time of a specific stream (video or audio)."""
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", stream_type,
        "-show_entries", "format=start_time",
        "-of", "json", media_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    metadata = json.loads(result.stdout)
    return float(metadata["format"]["start_time"]) if "format" in metadata else 0

def apply_start_time(row):
    try:
        media_path = row["path"]
        stream_type = "v:0" if row["filetype"] == "video" else "a:0"
        start_time = get_media_start_time(media_path, stream_type)
        return start_time
    except Exception as e:
        print(f"Error processing {row['path']}: {e}")
        return None
    

# test_recordings_df["start_time"] = test_recordings_df.apply(apply_start_time, axis=1)
# test_recordings_df

# recordings_df["start_time"] = recordings_df.apply(apply_start_time, axis=1)
# recordings_df

# Normalize Video Files

In [19]:
import os
import subprocess
import json
import numpy as np
import re
from collections import deque

def check_normalized_video(video_path, expected_resolution="1280x720", expected_fps=30):
    """
    Performs integrity checks on a normalized video file to ensure it is valid and meets expected properties.
    
    **Checks performed:**
    1. **File existence** - Ensures the file is present at the expected location.
    2. **File size check** - Ensures the file is not empty (zero bytes).
    3. **File readability** - Attempts to decode the file to check if it is playable.
    4. **Duration validity** - Ensures the duration is greater than 0.
    5. **Resolution match** - Checks if the video resolution matches the expected resolution.
    6. **Frame rate check** - Verifies that the frame rate is within an acceptable margin of error.
    7. **Frame integrity checks (via `showinfo`)**:
       - **Sequential Frames Check**: Ensures `n` values are strictly increasing.
       - **PTS Order Check**: Ensures frame timestamps (`pts_time`) increase monotonically.
       - **Duplicate Frames Detection**: Identifies repeated frames by comparing `checksum`.
       - **Frozen Frame Detection**: Flags extended sequences of frames with identical `mean` values.
    8. **Overall Validation** - Returns a single `True`/`False` flag indicating whether the file is usable.

    **Returns:**
    A dictionary containing:
    - Results of each check (`True`/`False`).
    - A `"is_valid"` flag that summarizes whether the file passes all critical checks.
    - An `"errors"` field for debugging if the file is not valid.
    """

    results = {
        "exists": os.path.exists(video_path),
        "file_size": None,  # File size in bytes
        "readable": False,
        "duration_valid": False,
        "resolution_valid": False,
        "fps_valid": False,
        "frame_numbers_in_order": False,
        "pts_in_order": False,
        "duplicated_frames": False,
        "frozen_frames": False,
        
        "errors": None,
        "is_valid": False,  # Overall validation flag
    }

    # 1. **Check if the file exists**
    if not results["exists"]:
        results["errors"] = "File not found."
        return results

    # 2. **Check if the file size is zero**
    file_size = os.path.getsize(video_path)
    results["file_size"] = file_size

    if file_size == 0:
        results["errors"] = "File exists but is empty (zero bytes)."
        return results  # Exit early

    # 3. **Check if the file is readable**
    cmd_readable = [
        "ffmpeg", "-v", "error", "-i", video_path, "-f", "null", "-"
    ]
    readable_process = subprocess.run(cmd_readable, capture_output=True, text=True)

    if readable_process.returncode == 0:
        results["readable"] = True
    else:
        results["errors"] = "File exists but is unreadable."
        return results  # Exit early

    # 4. **Extract video metadata (duration, resolution, fps)**
    cmd_metadata = [
        "ffprobe", "-v", "error", "-select_streams", "v",
        "-show_entries", "stream=width,height,r_frame_rate,duration",
        "-of", "json", video_path
    ]
    result = subprocess.run(cmd_metadata, capture_output=True, text=True)

    if result.returncode != 0:
        results["errors"] = "FFprobe failed to read file."
        return results

    try:
        metadata = json.loads(result.stdout)
        stream = metadata["streams"][0]

        # 5. **Check video duration**
        duration = float(stream.get("duration", 0))
        results["duration_valid"] = duration > 0  # Duration must be greater than zero

        # 6. **Check resolution**
        width, height = stream["width"], stream["height"]
        expected_width, expected_height = map(int, expected_resolution.split("x"))
        results["resolution_valid"] = (width, height) == (expected_width, expected_height)

        # 7. **Check frame rate**
        frame_rate = eval(stream["r_frame_rate"])  # Convert "30/1" to 30.0
        results["fps_valid"] = abs(frame_rate - expected_fps) < 0.5  # Allow small rounding error

    except Exception as e:
        results["errors"] = f"Error parsing metadata: {e}"
        return results

    # 8. **Extract Frame Data using showinfo**
    cmd_showinfo = [
        "ffmpeg", "-i", video_path, "-vf", "showinfo", "-f", "null", "-"
    ]
    result_showinfo = subprocess.run(cmd_showinfo, capture_output=True, text=True)

    if result_showinfo.returncode != 0:
        results["errors"] = "Failed to extract frame data."
        return results

    frame_pattern = re.compile(
        r"\[Parsed_showinfo_\d+ @ 0x[0-9a-fA-F]+\]\s+n:\s*(\d+)\s+"
        r"pts:\s*(-?\d+)\s+pts_time:\s*([\d.]+)\s+"
        r"duration:\s*(-?\d+)\s+duration_time:\s*([\d.]+).*?"
        r"checksum:([0-9A-F]+).*?\s+"
        r"plane_checksum:\[[0-9A-F\s]+\]\s*"
        r"mean:\[\s*(\d+)\s+(\d+)\s+(\d+)\s*\]"
    )

    frame_numbers = []
    pts_times = []
    checksums = []
    mean_values = []

    for line in result_showinfo.stderr.split("\n"):
        match = frame_pattern.search(line)
        if match:
            frame_numbers.append(int(match.group(1)))
            pts_times.append(float(match.group(3)))
            checksums.append(match.group(6))
            mean_values.append([int(match.group(7)), int(match.group(8)), int(match.group(9))])

    # **Check for Out-of-Order Frames**
    results["frame_numbers_in_order"] = all(x < y for x, y in zip(frame_numbers, frame_numbers[1:]))

    # **Check for PTS Order**
    results["pts_in_order"] = all(x < y for x, y in zip(pts_times, pts_times[1:]))

    # **Check for Duplicated Frames (Identical Checksums)**
    results["duplicated_frames"] = any(x == y for x, y in zip(checksums, checksums[1:]))

    # **Check for Frozen Frames (Extended Sequences of Identical Mean Values)**
    frozen_frame_threshold = 20  # Flag runs of 20+ identical frames
    tolerance = 1  # Allow small variations

    recent_means = deque(maxlen=frozen_frame_threshold)
    current_sequence = []
    frozen_sequences = []

    for i, mean in enumerate(mean_values):
        if recent_means and all(abs(m - n) <= tolerance for m, n in zip(recent_means[-1], mean)):
            current_sequence.append(frame_numbers[i])
        else:
            if len(current_sequence) >= frozen_frame_threshold:
                frozen_sequences.append(current_sequence)
            current_sequence = [frame_numbers[i]]

        recent_means.append(mean)

    if len(current_sequence) >= frozen_frame_threshold:
        frozen_sequences.append(current_sequence)

    results["frozen_frames"] = len(frozen_sequences) > 0

    # 9. **Set Overall Validation Flag**
    results["is_valid"] = all([
        results["exists"], results["file_size"] > 0, results["readable"],
        results["duration_valid"], results["resolution_valid"], results["fps_valid"],
        results["frame_numbers_in_order"], results["pts_in_order"],
        # not results["duplicated_frames"], not results["frozen_frames"]
    ])

    return results

# check_normalized_video("recordings/20250301_0011_demo_QPK9NM/1740787968485-353873af-6c54-4510-bbcd-bd275feb611c-cam-video-1740788033829.webm") # invalid, pre-normalized
# check_normalized_video("recordings/20250301_0011_demo_QPK9NM/1740787968485-353873af-6c54-4510-bbcd-bd275feb611c-cam-video-1740788033829_normalized_start_66046.mp4")
# check_normalized_video("recordings/20250301_0011_demo_QPK9NM/1740787968485-4395b77a-c5d1-482a-b413-ca212d06fc0c-cam-video-1740787968759_normalized_start_1193.mp4")

In [37]:
import json
import os
import subprocess

FINAL_RESOLUTION = "1280:720"
FRAME_RATE = 30
FONT_PATH = "/System/Library/Fonts/Supplemental/Arial.ttf"

def parse_mute_events(mute_events_str):
    """
    Parses a JSON string containing mute events into structured mute/unmute times.
    
    Returns:
    - Two lists of (mute_start, mute_end) tuples:
        - `video_mute_intervals` for video mutes
        - `audio_mute_intervals` for audio mutes
    """
    try:
        mute_events = json.loads(mute_events_str)  # Safely parse JSON
    except json.JSONDecodeError as e:
        print(f"❌ Error parsing mute events JSON: {e}")
        return [], []  # Return empty lists in case of error

    video_mute_intervals = []
    audio_mute_intervals = []
    active_video_mute = None
    active_audio_mute = None

    for event, timestamp in mute_events:
        if event == "muted-video":
            active_video_mute = timestamp  # Start video mute
        elif event == "unmuted-video" and active_video_mute is not None:
            video_mute_intervals.append((active_video_mute, timestamp))  # Store mute period
            active_video_mute = None  # Reset

        elif event == "muted-audio":
            active_audio_mute = timestamp  # Start audio mute
        elif event in ["unmuted-audio", "unmuted-audio-as-speaker"] and active_audio_mute is not None:
            audio_mute_intervals.append((active_audio_mute, timestamp))  # Store mute period
            active_audio_mute = None  # Reset

    # If a mute started but was never unmuted, assume it lasts to the end
    if active_video_mute is not None:
        video_mute_intervals.append((active_video_mute, "end"))
    if active_audio_mute is not None:
        audio_mute_intervals.append((active_audio_mute, "end"))

    return video_mute_intervals, audio_mute_intervals


def normalize_video(video_path, start_time, position, game, sampleId, muteEvents, add_info=False, overwrite=False):
    """
    Normalizes video: Ensures fixed resolution and frame rate, adds optional text overlays.
    
    Displays "Video Muted" and/or "Audio Muted" based on mute events.
    """
    intermediate_path = video_path.replace(".webm", "_intermediate_fixed.mp4")
    output_path = video_path.replace(".webm", f"_normalized_start_{int(1000 * start_time)}.mp4")

    # **Step 1: Pre-check if the file already exists and is valid**
    if not overwrite:
        check_result = check_normalized_video(output_path)
        if check_result["is_valid"]:
            print(f"👉 Video already normalized: {output_path}")
            return output_path, None, check_result  # Skip processing

    # **Step 2: Convert WebM to MP4, ensuring resolution is fixed**
    cmd1 = [
        "ffmpeg", "-y", "-hwaccel", "videotoolbox", "-threads", "0",
        "-i", video_path,
        "-vf", f"scale={FINAL_RESOLUTION}:flags=lanczos",
        "-c:v", "h264_videotoolbox", "-crf", str(FRAME_RATE), "-preset", "fast",
        intermediate_path
    ]

    result1 = subprocess.run(cmd1, capture_output=True, text=True)
    if result1.returncode != 0:
        error_message = result1.stderr.strip()
        print(f"❌ FFmpeg Error (Step 1) for {video_path}:\n{error_message}")
        return None, error_message, None

    # **Step 3: Convert MP4 to final MP4, enforcing CFR and adding text overlays**
    video_filters = [f"fps={FRAME_RATE}"]

    # **Add Metadata Text Overlay (If Requested)**
    if add_info:
        metadata_text = f"Sample Id {sampleId}\nGroup {game}\nPosition {position}"
        video_filters.append(
            f"drawtext=fontfile='{FONT_PATH}':"
            f"text='{metadata_text}':x=10:y=10:fontcolor=white:fontsize=24:box=1:boxcolor=black@0.5"
        )

    # **Parse Mute Events & Add Mute Overlays**
    video_mute_intervals, audio_mute_intervals = parse_mute_events(muteEvents)

    for start, end in video_mute_intervals:
        if end == "end":
            enable_filter = f"gte(t,{start})"  # Enable from start until the end
        else:
            enable_filter = f"between(t,{start},{end})"  # Enable for a specific time range

        video_filters.append(
            f"drawtext=fontfile='{FONT_PATH}':"
            f"text='Video Muted':x=10:y=90:fontcolor=red:fontsize=24:"
            f"enable='{enable_filter}'"
        )

    for start, end in audio_mute_intervals:
        if end == "end":
            enable_filter = f"gte(t,{start})"
        else:
            enable_filter = f"between(t,{start},{end})"

        video_filters.append(
            f"drawtext=fontfile='{FONT_PATH}':"
            f"text='Audio Muted':x=10:y=130:fontcolor=red:fontsize=24:"
            f"enable='{enable_filter}'"
        )

    # **Build FFmpeg Filter String**
    vf_filter = ",".join(video_filters)

    cmd2 = [
        "ffmpeg", "-y", "-threads", "0", "-hwaccel", "videotoolbox",
        "-i", intermediate_path,
        "-vf", vf_filter,
        "-c:v", "h264_videotoolbox", "-crf", str(FRAME_RATE), "-preset", "fast",
        output_path
    ]

    result2 = subprocess.run(cmd2, capture_output=True, text=True)
    if result2.returncode != 0:
        error_message = result2.stderr.strip()
        print(f"❌ FFmpeg Error (Step 2) for {video_path}:\n{error_message}")
        return None, error_message, None

    # **Step 4: Delete intermediate file**
    if os.path.exists(intermediate_path):
        os.remove(intermediate_path)
        print(f"🗑️ Deleted intermediate file: {intermediate_path}")

    # **Step 5: Validate Final Output**
    final_check = check_normalized_video(output_path)
    if not final_check["is_valid"]:
        print(f"⚠️ Warning: Final video {output_path} may be corrupted.")
        return None, "Final output validation failed", final_check

    print(f"✅ Successfully processed {output_path}")
    return output_path, None, final_check  # No errors

# row = recordings_df.iloc[6]
# normalize_video(
#     video_path=row['path'], 
#     start_time=row['start_time'], 
#     position=row['position'],
#     game=row['game'],
#     sampleId=row['sampleId'],
#     muteEvents=row['muteEvents'],
#     add_info=True, 
#     overwrite=True
# )

# Normalize Audio Files

In [21]:
import subprocess
import json
import os

def check_normalized_audio(audio_path):
    """
    Performs integrity checks on an audio file to ensure it is valid and meets expected properties.

    **Checks performed:**
    1. **File existence** - Ensures the file is present at the expected location.
    2. **File size check** - Ensures the file is not empty (zero bytes).
    3. **File readability** - Attempts to decode the file to check if it is playable.
    4. **Duration validity** - Ensures the duration is greater than 0.
    5. **Overall Validation** - Returns a single `True`/`False` flag indicating whether the file is usable.

    **Returns:**
    A dictionary containing:
    - Results of each check (`True`/`False`).
    - `"is_valid"` flag that summarizes whether the file passes all critical checks.
    - `"errors"` field for debugging if the file is not valid.
    """

    results = {
        "exists": os.path.exists(audio_path),
        "file_size": None,  # File size in bytes
        "readable": False,
        "duration_valid": False,
        "errors": None,
        "is_valid": False,  # Overall validation flag
    }

    # 1. **Check if the file exists**
    if not results["exists"]:
        results["errors"] = "File not found."
        return results

    # 2. **Check file size (ensure it's not empty)**
    file_size = os.path.getsize(audio_path)
    results["file_size"] = file_size

    if file_size == 0:
        results["errors"] = "File exists but is empty (zero bytes)."
        return results  # Exit early

    # 3. **Check if the file is readable**
    cmd_readable = [
        "ffmpeg", "-v", "error", "-i", audio_path, "-f", "null", "-"
    ]
    readable_process = subprocess.run(cmd_readable, capture_output=True, text=True)

    if readable_process.returncode == 0:
        results["readable"] = True
    else:
        results["errors"] = "File exists but is unreadable."
        return results  # Exit early

    # 4. **Extract audio metadata (duration check)**
    cmd_metadata = [
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "json", audio_path
    ]
    result = subprocess.run(cmd_metadata, capture_output=True, text=True)

    if result.returncode != 0:
        results["errors"] = "FFprobe failed to read file."
        return results

    try:
        metadata = json.loads(result.stdout)
        duration = float(metadata["format"]["duration"]) if "format" in metadata and "duration" in metadata["format"] else 0
        results["duration_valid"] = duration > 0  # Duration must be greater than zero
    except Exception as e:
        results["errors"] = f"Error parsing metadata: {e}"
        return results

    # 5. **Set Overall Validation Flag**
    # The file is valid if all critical checks pass
    results["is_valid"] = (
        results["exists"]
        and results["file_size"] > 0
        and results["readable"]
        and results["duration_valid"]
    )

    return results

# Example usage
# check_normalized_audio("recordings/20250301_0011_demo_QPK9NM/1740787968485-4395b77a-c5d1-482a-b413-ca212d06fc0c-cam-audio-1740787968764_normalized_start_240.mp3")

In [30]:
import os
import subprocess

def normalize_audio(audio_path, start_time, overwrite=False):
    """
    Normalize an audio file: Convert Opus to MP3, adjust timing so the audio starts at 0, and force mono.

    **Steps:**
    1. Check if a valid normalized file already exists.
    2. If overwrite is `False` and the file is valid, return existing file.
    3. Convert WebM audio to MP3, apply timing correction, and force mono.
    4. Validate the output to ensure it was correctly processed.
    
    **Returns:**
    - If successful: `(output_path, None)`
    - If failed: `(None, error_message)`
    """

    # Define output file path
    output_path = audio_path.replace(".webm", f"_normalized_start_{int(1000 * start_time)}.mp3")

    # **Check if the file is already valid**
    if not overwrite:
        audio_check = check_normalized_audio(output_path)
        if audio_check["is_valid"]:
            print(f"👉 Audio already normalized: {output_path}")
            return output_path, None, None  # Skip reprocessing

    # **FFmpeg command to normalize audio**
    cmd = [
        "ffmpeg", "-y", 
        "-itsoffset", f"-{start_time}", "-i", audio_path,  # Shift start time to 0
        "-af", "pan=mono|c0=0.5*c0+0.5*c1",  # Convert stereo to mono
        "-c:a", "libmp3lame", "-b:a", "192k", output_path
    ]

    print(f"🔄 Processing audio: {audio_path}")
    result = subprocess.run(cmd, capture_output=True, text=True)

    # **Check for errors**
    if result.returncode != 0:
        error_message = result.stderr.strip()
        print(f"❌ FFmpeg Error for {audio_path}:\n{error_message}")
        return None, error_message, None # Return failure

    # **Validate the output file**
    final_check = check_normalized_audio(output_path)
    if not final_check["is_valid"]:
        print(f"❌ Audio validation failed after processing: {output_path}")
        return None, "Post-processing validation failed.", final_check

    print(f"✅ Audio successfully normalized: {output_path}")
    return output_path, None, final_check  # Success

# Example usage
# normalize_audio("recordings/20250301_0011_demo_QPK9NM/1740787968485-4395b77a-c5d1-482a-b413-ca212d06fc0c-cam-audio-1740787968764.webm", 0.24)

In [23]:
def get_media_duration(file_path):
    """Extracts duration of a media file (audio or video) using ffprobe."""

    if not isinstance(file_path, str) or not os.path.exists(file_path):  # Check if file path is valid
        print(f"Invalid file path: {file_path}")
        return None  # Skip invalid file paths

    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "json", file_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    try:
        metadata = json.loads(result.stdout)
        return float(metadata["format"]["duration"])  # Duration in seconds
    except (json.JSONDecodeError, KeyError, TypeError):
        return None
    
# recordings_df["duration"] = recordings_df.apply(lambda row: get_media_duration(row["normalized_path"]), axis=1)
# recordings_df["end_time"] = recordings_df["start_time"] + recordings_df["duration"]
# recordings_df   

# test_recordings_df["duration"] = test_recordings_df.apply(lambda row: get_media_duration(row["normalized_path"]), axis=1)
# test_recordings_df["end_time"] = test_recordings_df["start_time"] + test_recordings_df["duration"]
# test_recordings_df

In [24]:
import pandas as pd

def check_overlapping_files(group_df):
    """
    Checks that within each group:
    - No video files at the same position overlap.
    - No audio files at the same position overlap.
    """

    # Sort by position, file type, and start time
    group_df = group_df.sort_values(by=["position", "filetype", "start_time"]).reset_index(drop=True)

    # Track overlap warnings
    overlap_warnings = []

    # Iterate over unique positions
    for position in group_df["position"].unique():
        for filetype in ["video", "audio"]:
            # Filter by position and file type
            subset_df = group_df[(group_df["position"] == position) & (group_df["filetype"] == filetype)].copy()

            # Compute end times
            subset_df["end_time"] = subset_df["start_time"] + subset_df["duration"]

            # Check for overlaps
            for i in range(1, len(subset_df)):
                prev_row = subset_df.iloc[i - 1]
                curr_row = subset_df.iloc[i]

                if curr_row["start_time"] < prev_row["end_time"]:
                    overlap_warnings.append(
                        f"❌ Warning: Overlapping {filetype} files detected at position {position} in group {group_df.iloc[0]['group']}.\n"
                        f"Files: {prev_row['filename']} ({prev_row['start_time']} - {prev_row['end_time']}) and "
                        f"{curr_row['filename']} ({curr_row['start_time']} - {curr_row['end_time']})."
                    )

    # Print warnings if any exist
    if overlap_warnings:
        print("\n".join(overlap_warnings))

# Apply the function to each group
# recordings_df = recordings_df.groupby("group", group_keys=False).apply(check_overlapping_files).reset_index(drop=True)

# just first 3 groups
# num_groups = 3  # Change this to however many groups you want
# first_n_groups = recordings_df["group"].unique()[:num_groups]

# (recordings_df[recordings_df["group"].isin(first_n_groups)]
#     .groupby("group", group_keys=False)
#     .apply(check_overlapping_files)
# )

# # apply to test video
# test_recordings_df.groupby("group", group_keys=False).apply(check_overlapping_files)


In [26]:
def generate_black_video(black_video_path, final_resolution, frame_rate, duration):
    """Check if black background video exists; if not, generate it."""
    if os.path.exists(black_video_path):
        print(f"Black video already exists: {black_video_path}")
        return  # Skip generation

    print(f"Generating black video: {black_video_path}")
    cmd = [
        "ffmpeg",
        "-f", "lavfi",
        "-i", f"color=c=black:s={final_resolution}:r={frame_rate}",
        "-t", str(duration),
        "-c:v", "libx264",
        "-preset", "fast",
        "-y", black_video_path  # Overwrite only if missing
    ]

    subprocess.run(cmd, check=True)



# generate_black_video(black_video_path="recordings/black_2min.mp4", final_resolution="2560x720", frame_rate=30, duration=120)
generate_black_video(black_video_path="recordings/black_15min.mp4", final_resolution="2560x720", frame_rate=30, duration=900)

Generating black video: recordings/black_15min.mp4


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex -

In [27]:
import pandas as pd
import subprocess
import os

# Constants
BLACK_VIDEO_PATH = "recordings/black.mp4"  # Pre-generated black background
FINAL_RESOLUTION = "2560:720"  # Ensure it fits side-by-side layout
final_width, final_height = map(int, FINAL_RESOLUTION.split(":"))

def merge_videos_for_group(group_df, black_video_path=BLACK_VIDEO_PATH):
    """
    Merge participant videos onto a black background, aligning them based on offsets.
    - Ensures position 0 is always on the left.
    - Ensures position 1 is always on the right.
    - Makes each video disappear instead of holding the last frame.
    """

    group_id = group_df.iloc[0]["group"]
    mtgStartTime = group_df.iloc[0]["mtgStartTime"]
    output_path = os.path.join("recordings", group_df.iloc[0]["folder"], f"{mtgStartTime}_merged_video.mp4")

    # **Filter to only video files** (Exclude audio files)
    video_files = group_df[group_df["filetype"] == "video"]

    if video_files.empty:
        print(f"Skipping session {group_id} - No video files found.")
        return None

    # Ensure we have exactly two positions: left (0) and right (1)
    positions = sorted(video_files["position"].dropna().unique())
    if positions != [0, 1]:
        print(f"Skipping session {group_id} - Expected exactly 2 positions (0 and 1), found {positions}.")
        return None

    # Assign fixed positions
    position_map = {
        0: "(W/2-w)/2:(H-h)/2",      # Left position
        1: "(W/2-w)/2+W/2:(H-h)/2"   # Right position
    }

    # FFmpeg input files
    inputs = ["-i", black_video_path]  # Start with black background
    overlays = []
    video_index = 1  # Video input index in FFmpeg (excluding black background)

    for _, row in video_files.iterrows():
        if pd.isna(row["normalized_path"]):  # Skip missing videos
            continue

        start_time = row["start_time"]
        duration = row["duration"]  # Assuming you have the duration of each video
        position = position_map[row["position"]]  # Get left/right position

        # Add video input
        inputs.extend(["-i", row["normalized_path"]])

        # Scale video and adjust timestamp for correct alignment
        overlays.append(f"[{video_index}:v] setpts=PTS-STARTPTS+{start_time}/TB, scale={final_width//2}:-1 [v{video_index}];")

        # Ensure video disappears after its duration
        enable_filter = f"enable='between(t,{start_time},{start_time + duration})'"

        if video_index == 1:
            overlays.append(f"[0:v][v{video_index}] overlay={position}:{enable_filter} [out{video_index}];")
        else:
            overlays.append(f"[out{video_index-1}][v{video_index}] overlay={position}:{enable_filter} [out{video_index}];")

        video_index += 1

    # Build FFmpeg filter complex
    final_output_tag = f"[out{video_index-1}]"
    drawtext_filter = (
        f"{final_output_tag} drawtext=fontfile='{FONT_PATH}':"
        f"text='%{{pts\\:hms}}':"
        f"x=10: y=h-th-10: fontcolor=white: fontsize=20: box=1: boxcolor=black@0.5"
        f" [final_output]"
    )


    # filter_complex = " ".join(overlays)
    filter_complex = " ".join(overlays) + drawtext_filter

    cmd = [
        "ffmpeg", "-threads", "0",
        "-hwaccel", "videotoolbox",  # ✅ Use Mac GPU for decoding
        "-threads", "0",  # ✅ Allow FFmpeg to optimize CPU usage
        *inputs,
        "-filter_complex", filter_complex,
        # "-map", final_output_tag,  # Ensure only the final video stream is mapped
         "-map", "[final_output]",  # Ensure only the final video stream is mapped

        # "-c:v", "libx264", 
        "-c:v", "h264_videotoolbox",  # 🚀 Enable GPU acceleration
        "-b:v", "5M",  # Required for GPU encoding
        "-preset", "fast",
        "-y", output_path  # Overwrite existing file
    ]

    # Run FFmpeg
    print(f"Processing merged video for session {group_id}...")
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.returncode != 0:
        error_message = result.stderr.strip()
        print(f"❌ FFmpeg Error for {output_path}:\n{error_message}")
        return None

    # print(f" Successfully processed {output_path}")
    return output_path

# Apply function to a test group
# merge_videos_for_group(test_recordings_df, black_video_path="recordings/black_2min.mp4")  # Run the function on the first group only


# # just first 3 groups
# num_groups = 100  # Change this to however many groups you want
# first_n_groups = recordings_df["group"].unique()[:num_groups]

# (recordings_df[recordings_df["group"].isin(first_n_groups)]
#     .groupby("group", group_keys=False)
#     .apply(merge_videos_for_group)
# )

# n_groups = 250  # Adjust as needed

# for name, group in recordings_df.groupby("group"):
#     if group['normalized_path'].isnull().sum() > 0:
#         print(f"Skipping group {name} - missing normalized files.")
#         continue
#     try:
#         merge_videos_for_group(group)
#         print(f"✅ Processed group {index+1}/{n_groups}")
#     except Exception as e:
#         print(f"❌ Error processing row {index+1}: {e}")

# print("✅ Processing complete.")


In [ ]:
def generate_silence_if_needed(silence_audio_path, duration):
    """Generate a 15-minute silent MP3 file if it doesn't exist."""
    if not os.path.exists(silence_audio_path):
        print("Generating 15-minute silence file...")
        cmd = [
            "ffmpeg", "-f", "lavfi", "-t", str(duration), "-i", "anullsrc=r=48000:cl=stereo",
            "-c:a", "libmp3lame", "-b:a", "192k", silence_audio_path
        ]
        subprocess.run(cmd, check=True)
    else:
        print("Silence file already exists.")


# generate_silence_if_needed("recordings/silence_15min.mp3", 900)
generate_silence_if_needed("recordings/silence_2min.mp3", 120)

Generating 15-minute silence file...


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex -

In [ ]:
# this mostly works?
import pandas as pd
import subprocess
import os
import json

# Constants
SILENCE_AUDIO_PATH = "recordings/silence.mp3"  # Pre-generated silent audio file
FINAL_AUDIO_BITRATE = "192k"

def is_mono(audio_path):
    """Check if an audio file is mono using ffprobe."""
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "a:0",
        "-show_entries", "stream=channels", "-of", "json", audio_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    try:
        metadata = json.loads(result.stdout)
        return metadata["streams"][0]["channels"] == 1  # True if mono
    except (json.JSONDecodeError, KeyError, IndexError):
        return False  # Assume not mono if any error occurs

def merge_audio_for_group(group_df, silence_audio_path=SILENCE_AUDIO_PATH, duration=900):
    """
    Merge participant audio files into a stereo track:
    - First participant -> Left channel (c0)
    - Second participant -> Right channel (c1)
    - Properly aligns based on start offsets
    """
    group_id = group_df.iloc[0]["group"]
    mtgStartTime = group_df.iloc[0]["mtgStartTime"]
    output_path = os.path.join("recordings", group_df.iloc[0]["folder"], f"{mtgStartTime}_merged_audio.mp3")

    # **Filter to only audio files** (Exclude video files)
    audio_files = group_df[group_df["filetype"] == "audio"]

    if audio_files.empty:
        print(f"Skipping session {group_id} - No audio files found.")
        return None

    # Get unique participant IDs
    participants = sorted(audio_files["position"].dropna().unique())

    if len(participants) != 2:
        print(f"Skipping session {group_id} - Expected 2 participants, found {len(participants)}.")
        return None

    # Assign left (P1) and right (P2) channels
    participant_channels = {
        participants[0]: "c0",  # Left channel
        participants[1]: "c1"   # Right channel
    }

    # FFmpeg input files
    inputs = ["-i", silence_audio_path]  # Start with silent audio
    audio_filters = ["[0:a] acopy [a0];"]
    audio_index = 1  # Audio input index in FFmpeg

    for _, row in audio_files.iterrows():
        if pd.isna(row["normalized_path"]):  # Skip missing audio
            continue

        audio_path = row["normalized_path"]

        # **Check if audio is mono before proceeding**
        if not is_mono(audio_path):
            print(f"Error: {audio_path} is not mono. Please re-run normalization.")
            return None  # Stop processing if an issue is found

        start_time = row["start_time"]
        participant_id = row["position"]
        channel = participant_channels[participant_id]

        # Add audio input
        inputs.extend(["-i", audio_path])

        # Offset audio to align correctly and map to left or right
        end_pad = (duration-row["end_time"])*48000
        # audio_filters.append(f"[{audio_index}:a] adelay={int(start_time*1000)}, apad=pad_len={end_pad}, pan=stereo|{channel}=c0 [a{audio_index}];")
        # audio_filters.append(f"[{audio_index}:a] adelay={int(start_time*1000)}, apad=pad_len={end_pad}, pan=stereo|{channel}=0.5*c0+0.5*c1 [a{audio_index}];")
        audio_filters.append(f"[{audio_index}:a] adelay={int(start_time*1000)}, apad=pad_len={end_pad} [a{audio_index}];") # keep as mono
        audio_index += 1

    # Merge into a stereo track
    audio_tracks = [f"[a{i}]" for i in range(audio_index)]
    merged_audio = f"{''.join(audio_tracks)} amerge=inputs={len(audio_files)+1} [final_out];"

    # Build FFmpeg filter complex
    filter_complex = " ".join(audio_filters) + merged_audio
    final_output_tag = "[final_out]"

    cmd = [
        "ffmpeg",
        *inputs,
        "-filter_complex", filter_complex,
        "-map", final_output_tag,  # Ensure only the final audio stream is mapped
        "-c:a", "libmp3lame", "-b:a", FINAL_AUDIO_BITRATE,
        "-y", output_path  # Overwrite existing file
    ]

    # Run FFmpeg
    print(f"Processing merged audio for session {group_id}...")
    # print(cmd)

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.returncode != 0:
        error_message = result.stderr.strip()
        print(f"❌ FFmpeg Error for {output_path}:\n{error_message}")
        return None

    return output_path

# merge_audio_for_group(test_recordings_df, silence_audio_path="recordings/silence_2min.mp3", duration=120)  # Run the function on the first group only

# for name, group in recordings_df.groupby("group"):
#     if group['normalized_path'].isnull().sum() > 0:
#         print(f"👉 Skipping group {name} - missing normalized files.")
#         continue
#     try:
#         merge_audio_for_group(group)
#         print(f"✅ Processed group {name}")
#     except Exception as e:
#         print(f"❌ Error processing row {index+1}: {e}")

# print("✅ Processing complete.")

In [ ]:
import subprocess
import os
import shutil

# Define the destination folder for finalized files
READY_TO_CODE_DIR = "recordings/ready_to_code"

# Ensure the directory exists
os.makedirs(READY_TO_CODE_DIR, exist_ok=True)

def merge_audio_video_for_group(group_df, overwrite=False):
    """
    Merge the group's merged video and audio files into a final MP4 file.
    - Skips if the final output already exists.
    - Saves a copy in 'recordings/ready_to_code/' with a detailed filename.
    """
    group_id = group_df.iloc[0]["group"]
    mtgStartTime = group_df.iloc[0]["mtgStartTime"]
    date = group_df.iloc[0]["date"]
    time = group_df.iloc[0]["time"]
    batch = group_df.iloc[0].get("batch", "unknown")  # Ensure batch exists
    game_number = group_df.iloc[0].get("game", "unknown")  # Ensure game number exists
    folder_path = os.path.join("recordings", group_df.iloc[0]["folder"])

    # Define expected input/output file paths
    video_path = os.path.join(folder_path, f"{mtgStartTime}_merged_video.mp4")
    audio_path = os.path.join(folder_path, f"{mtgStartTime}_merged_audio.mp3")
    new_filename = f"{date}_{time}_batch_{batch}_group_{group_id}_game_{game_number}.mp4"
    output_path = os.path.join(folder_path, new_filename)
    

    # Check if the final output already exists
    if not overwrite and os.path.exists(output_path):
        print(f"✔ Skipping group {group_id} - Final output already exists: {output_path}")
    else:
        # Check if required files exist before proceeding
        if not os.path.exists(video_path):
            print(f"❌ Skipping session {group_id} - Merged video not found: {video_path}")
            return None

        if not os.path.exists(audio_path):
            print(f"❌ Skipping session {group_id} - Merged audio not found: {audio_path}")
            return None

        # FFmpeg command to merge audio and video
        cmd = [
            "ffmpeg",
            "-i", video_path,  # Input video
            "-i", audio_path,  # Input audio
            "-map", "0:v",  # Ensure we take video from first input
            "-map", "1:a",  # Ensure we take audio from second input
            "-c:v", "copy",  # Don't re-encode video
            "-c:a", "aac", "-b:a", "192k",  # Convert audio to AAC (for MP4 compatibility)
            "-y", output_path  # Overwrite if exists
        ]

        # Run FFmpeg
        print(f"Processing final merged file for session {group_id}...")
        print(f"\n\nCMD:\n{' '.join(cmd)}\n\n")
        subprocess.run(cmd, check=True)
        print(f"✔ Merged {group_id}: {output_path}")

    # Copy the finalized file
    destination_path = os.path.join(READY_TO_CODE_DIR, new_filename)
    shutil.copy(output_path, destination_path)
    print(f"📂 Copied to: {destination_path}")

    return output_path



In [ ]:
# import subprocess
# import os

# def merge_audio_video_for_group(group_df):
#     """
#     Merge the group's merged video and audio files into a final MP4 file.
#     Uses the same folder and file naming convention as previous steps.
#     """
#     group_id = group_df.iloc[0]["group"]
#     mtgStartTime = group_df.iloc[0]["mtgStartTime"]
#     folder_path = os.path.join("recordings", group_df.iloc[0]["folder"])

#     # Define expected input/output file paths
#     video_path = os.path.join(folder_path, f"{mtgStartTime}_merged_video.mp4")
#     audio_path = os.path.join(folder_path, f"{mtgStartTime}_merged_audio.mp3")
#     output_path = os.path.join(folder_path, f"{mtgStartTime}_final_output.mp4")

#     # Check if required files exist
#     if not os.path.exists(video_path):
#         print(f"Skipping session {group_id} - Merged video not found: {video_path}")
#         return None

#     if not os.path.exists(audio_path):
#         print(f"Skipping session {group_id} - Merged audio not found: {audio_path}")
#         return None

#     # FFmpeg command to merge audio and video
#     cmd = [
#         "ffmpeg",
#         "-i", video_path,  # Input video
#         "-i", audio_path,  # Input audio
#         "-map", "0:v",  # Ensure we take video from first input
#         "-map", "1:a",  # Ensure we take audio from second input
#         "-c:v", "copy",  # Don't re-encode video
#         "-c:a", "aac", "-b:a", "192k",  # Convert audio to AAC (for MP4 compatibility)
#         "-y", output_path  # Overwrite if exists
#     ]

#     # Run FFmpeg
#     print(f"Processing final merged file for session {group_id}...")
#     print(f"\n\nCMD:\n{' '.join(cmd)}\n\n")

#     result = subprocess.run(cmd, check=True, capture_output=True, text=True)
#     if result.returncode != 0:
#         error_message = result.stderr.strip()
#         print(f"❌ FFmpeg Error for {output_path}:\n{error_message}")
#         return None

#     return output_path


# # apply to test video
# # merge_audio_video_for_group(test_recordings_df)


In [53]:
import datetime
overwrite = False # whether to overwrite normalized files. (Always overwrites merged files)


groups = [409]

for name, group in recordings_df[recordings_df["game"].isin(groups)].groupby("group"):
    print(f"\n\nProcessing group {name}...")
    start = datetime.datetime.now()
    if group['discussion_happened'].all() == False:
        print(f"Skipping group {name} - discussion did not happen.")
        continue

    try:
        print("Get start times")
        group["start_time"] = group.apply(apply_start_time, axis=1)

        print("Normalize Video and Audio")
        for index in group.index:
            row = group.loc[index]
            

            if row["filetype"] == "video":
                normalized_path, error_msg, checks = normalize_video(row["path"], float(row["start_time"]), row["position"], row["game"], row["sampleId"], muteEvents=row["muteEvents"], add_info=True, overwrite=overwrite)
                group.at[index, "normalized_path"] = normalized_path
                group.at[index, "error_msg"] = error_msg
                group.at[index, "duplicated_frames"] = checks["duplicated_frames"]
                group.at[index, "frozen_frames"] = checks["frozen_frames"]
                group.at[index, "is_valid"] = checks["is_valid"]
                if error_msg:
                    print(f"❌ Error processing video {row['path']}: {error_msg}")
       
            elif row["filetype"] == "audio":
                normalized_path, error_msg, checks = normalize_audio(row["path"], float(row["start_time"]), overwrite=overwrite)
                group.at[index, "normalized_path"] = normalized_path
                group.at[index, "error_msg"] = error_msg
                recordings_df.at[index, "duplicated_frames"] = pd.NA
                recordings_df.at[index, "frozen_frames"] = pd.NA
                group.at[index, "is_valid"] = check_normalized_audio(normalized_path)["is_valid"]
                if error_msg:
                    print(f"❌ Error processing audio {row['path']}: {error_msg}")

        print("Get audio durations")
        group["duration"] = group.apply(lambda row: get_media_duration(row["normalized_path"]), axis=1)
        group["end_time"] = group["start_time"] + group["duration"]

        print("Check overlapping files")
        check_overlapping_files(group)

        print("merge videos")
        merge_videos_for_group(group, black_video_path="recordings/black.mp4")

        print("merge audio")
        merge_audio_for_group(group, silence_audio_path="recordings/silence.mp3", duration=900)

        print("merge audio video")
        merge_audio_video_for_group(group)

        end = datetime.datetime.now()
        duration = end-start
        total_seconds = duration.total_seconds()
        minutes = int(total_seconds // 60)
        seconds = int(total_seconds % 60)

        print(f"✅ Processed group {name} in {minutes}:{seconds}")
    except Exception as e:
        print(f"❌ Error processing group {name}: {e}")






Processing group WT9T50...
Get start times
Normalize Video and Audio
👉 Audio already normalized: recordings/20241219_1734_w2s26_WT9T50/1734631629271-0169825f-b8b5-45dc-8c60-6fd042341208-cam-audio-1734632492892_normalized_start_863424.mp3
👉 Video already normalized: recordings/20241219_1734_w2s26_WT9T50/1734631629271-0169825f-b8b5-45dc-8c60-6fd042341208-cam-video-1734632493116_normalized_start_865266.mp4
👉 Audio already normalized: recordings/20241219_1734_w2s26_WT9T50/1734631629271-09cf2b10-ae44-4fd9-941c-f51cbb73d5ee-cam-audio-1734631975159_normalized_start_345680.mp3
👉 Video already normalized: recordings/20241219_1734_w2s26_WT9T50/1734631629271-09cf2b10-ae44-4fd9-941c-f51cbb73d5ee-cam-video-1734631975494_normalized_start_346863.mp4
👉 Audio already normalized: recordings/20241219_1734_w2s26_WT9T50/1734631629271-1482008a-401f-4e4d-ad04-509f6165a0b7-cam-audio-1734631630451_normalized_start_970.mp3
👉 Video already normalized: recordings/20241219_1734_w2s26_WT9T50/1734631629271-1482008

In [41]:
group.T

,2295,2296,2297,2298
path,recordings/20241219_1734_w2s26_ZK6YGD/1734631999615-1d3ce1f0-c7d4-4020-ba89-4dd962bf1982-cam-audio-1734632005066.webm,recordings/20241219_1734_w2s26_ZK6YGD/1734631999615-1d3ce1f0-c7d4-4020-ba89-4dd962bf1982-cam-video-1734632005294.webm,recordings/20241219_1734_w2s26_ZK6YGD/1734631999615-6fd1ab48-443c-4f05-adfe-055e59188840-cam-audio-1734632000690.webm,recordings/20241219_1734_w2s26_ZK6YGD/1734631999615-6fd1ab48-443c-4f05-adfe-055e59188840-cam-video-1734632000693.webm
folder,20241219_1734_w2s26_ZK6YGD,20241219_1734_w2s26_ZK6YGD,20241219_1734_w2s26_ZK6YGD,20241219_1734_w2s26_ZK6YGD
filename,1734631999615-1d3ce1f0-c7d4-4020-ba89-4dd962bf1982-cam-audio-1734632005066.webm,1734631999615-1d3ce1f0-c7d4-4020-ba89-4dd962bf1982-cam-video-1734632005294.webm,1734631999615-6fd1ab48-443c-4f05-adfe-055e59188840-cam-audio-1734632000690.webm,1734631999615-6fd1ab48-443c-4f05-adfe-055e59188840-cam-video-1734632000693.webm
date,20241219,20241219,20241219,20241219
time,1734,1734,1734,1734
mtgStartTime,1734631999615,1734631999615,1734631999615,1734631999615
sessionId,1d3ce1f0_c7d4_4020_ba89_4dd962bf1982,1d3ce1f0_c7d4_4020_ba89_4dd962bf1982,6fd1ab48_443c_4f05_adfe_055e59188840,6fd1ab48_443c_4f05_adfe_055e59188840
batch,w2s26,w2s26,w2s26,w2s26
group,ZK6YGD,ZK6YGD,ZK6YGD,ZK6YGD
filetype,audio,video,audio,video


In [36]:
group.index

Index([2295, 2296, 2297, 2298, 0, 1, 2, 3], dtype='int64')